# 01a — Stage Charades once (ONLINE, **CPU — do not attach a GPU**)

| attach as input | produces |
|---|---|
| nothing | `charades-480p` (~13 GB, **PRIVATE**) |

Pure I/O: downloads the Charades 480p archive and its annotations and republishes them as
a Kaggle dataset. **Set the accelerator to None.** Nothing here touches a GPU, and a GPU
session spent downloading is the exact waste this notebook exists to remove — extraction
previously re-fetched 13 GB at the start of each of 3-4 GPU sessions, ~30 min of idle card
every time.

Once this dataset exists, notebook 01 can run with the internet OFF, which means shard
extraction moves onto the Blackwell instead of a T4.

**Licence:** Charades forbids redistribution. Keep this dataset **private** — it is your
own working copy, exactly like the derived shard datasets.

The archives are stored as-is rather than unpacked. A single 13 GB file republishes far
faster than 9,848 individual mp4s, and notebook 01 unzips only the slice it needs.

In [ ]:
# Download to /kaggle/working so Save Version picks it up. The 480p archive is ~13 GB
# and working is capped at 20 GB, so the zips are stored WITHOUT unpacking - unpacking
# here would need ~26 GB and blow the cap.
import urllib.request, pathlib, time

OUT = pathlib.Path("/kaggle/working"); OUT.mkdir(parents=True, exist_ok=True)
SOURCES = [
    ("https://ai2-public-datasets.s3-us-west-2.amazonaws.com/charades/Charades_v1_480.zip",
     "Charades_v1_480.zip"),
    ("https://ai2-public-datasets.s3-us-west-2.amazonaws.com/charades/Charades.zip",
     "Charades_annotations.zip"),
]
for url, name in SOURCES:
    dst = OUT / name
    if dst.exists():
        print(f"{name}: already present ({dst.stat().st_size/1e9:.1f} GB)")
        continue
    t0 = time.time()
    print(f"downloading {name} ...")
    urllib.request.urlretrieve(url, dst)
    print(f"  {dst.stat().st_size/1e9:.1f} GB in {(time.time()-t0)/60:.1f} min")

In [ ]:
# Verify before publishing. A truncated download produces a zip that opens fine at the
# header and fails halfway through extraction - four hours into an offline session, with
# no way to re-fetch. testzip() reads every member's CRC, so it catches that here.
import zipfile, pathlib

OUT = pathlib.Path("/kaggle/working")
for name, expect_min_gb in (("Charades_v1_480.zip", 10.0),
                            ("Charades_annotations.zip", 0.001)):
    p = OUT / name
    gb = p.stat().st_size / 1e9
    assert gb >= expect_min_gb, f"{name} is {gb:.2f} GB - truncated download"
    with zipfile.ZipFile(p) as z:
        bad = z.testzip()
        assert bad is None, f"{name}: corrupt member {bad}"
        members = z.namelist()
    print(f"  {name:<28} {gb:>5.1f} GB, {len(members)} members, CRC ok")

with zipfile.ZipFile(OUT / "Charades_v1_480.zip") as z:
    vids = [n for n in z.namelist() if n.endswith(".mp4")]
assert len(vids) > 9000, f"only {len(vids)} mp4s - expected ~9,848"
print(f"\n{len(vids)} videos ready")
print("Save Version -> create a PRIVATE dataset named charades-480p")
print("Then notebook 01 runs OFFLINE: attach this + behaviorsense-code + the wheels.")